# Quick-Start 00: Your First RL Agent

**Estimated time: under 2 minutes**

## Learning Objectives

By the end of this notebook you will:
1. Understand the core RL loop: observation → action → reward → next observation
2. See why random action selection fails and how a greedy policy improves performance
3. Visualize reward accumulation across episodes

## Prerequisites

- Basic Python and NumPy
- No prior RL knowledge required

## No External Dependencies

This notebook uses only NumPy and Matplotlib — no gymnasium needed here. We build a tiny
4x4 gridworld from scratch so you can see every moving part.

---

## Setup

We import only NumPy and Matplotlib. The random seed ensures your results match the
discussion below — change it later to see how luck affects early learning.

In [ ]:
# Purpose: Imports and reproducibility
# Key Concept: Fixing a seed lets you reproduce exact trajectories for debugging

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from collections import defaultdict

np.random.seed(42)

print("Setup complete. NumPy version:", np.__version__)

## The Environment: A 4x4 Gridworld

The agent starts at the top-left corner `(0, 0)` and must reach the goal at the
bottom-right corner `(3, 3)`. One cell is a hole — falling in ends the episode with a
penalty.

```
 S  .  .  .
 .  .  H  .
 .  .  .  .
 .  .  .  G
```

**Actions:** 0=Up, 1=Down, 2=Left, 3=Right  
**Rewards:** +10 for reaching G, -5 for falling into H, -0.1 per step (encourages efficiency)  
**Episode ends:** on reaching G or H, or after 50 steps

This is the simplest possible environment where you can still see meaningful behavior.
Real RL environments (CartPole, Atari, robotics) follow exactly this same contract.

In [ ]:
# Purpose: Define the gridworld environment
# Key Concept: An RL environment exposes reset() and step(action) — that's the entire API

class GridWorld:
    """
    4x4 grid with a start, a goal, and one hole.

    The environment contract every RL env follows:
      - reset() -> state
      - step(action) -> (next_state, reward, done)
    """

    ACTIONS = {0: (-1, 0), 1: (1, 0), 2: (0, -1), 3: (0, 1)}  # Up Down Left Right
    ACTION_NAMES = {0: 'Up', 1: 'Down', 2: 'Left', 3: 'Right'}

    def __init__(self, size=4):
        self.size = size
        self.start = (0, 0)
        self.goal  = (size - 1, size - 1)
        self.hole  = (1, 2)          # one hazard cell
        self.max_steps = 50
        self.state = None
        self.steps = 0

    def reset(self):
        """Return the starting state."""
        self.state = self.start
        self.steps = 0
        return self.state

    def step(self, action):
        """
        Apply action, return (next_state, reward, done).

        Walls bounce the agent back to its current cell.
        """
        self.steps += 1
        dr, dc = self.ACTIONS[action]
        r, c = self.state

        # Step 1: Compute proposed next cell, clamp to grid boundaries (wall bounce)
        nr = max(0, min(self.size - 1, r + dr))
        nc = max(0, min(self.size - 1, c + dc))
        self.state = (nr, nc)

        # Step 2: Compute reward and check terminal conditions
        if self.state == self.goal:
            return self.state, +10.0, True
        elif self.state == self.hole:
            return self.state, -5.0, True
        elif self.steps >= self.max_steps:
            return self.state, -0.1, True   # timeout
        else:
            return self.state, -0.1, False  # step penalty

    def render(self, ax=None):
        """Draw the grid on a matplotlib Axes."""
        show = ax is None
        if ax is None:
            _, ax = plt.subplots(figsize=(4, 4))

        grid = np.zeros((self.size, self.size))
        goal_r, goal_c = self.goal
        hole_r, hole_c = self.hole
        agent_r, agent_c = self.state if self.state else self.start

        grid[goal_r, goal_c] = 2
        grid[hole_r, hole_c] = -1
        grid[agent_r, agent_c] = 1

        ax.imshow(grid, cmap='RdYlGn', vmin=-1, vmax=2)
        ax.set_xticks(range(self.size))
        ax.set_yticks(range(self.size))
        ax.set_xticklabels(range(self.size))
        ax.set_yticklabels(range(self.size))

        labels = {self.start: 'S', self.goal: 'G', self.hole: 'H'}
        if self.state and self.state not in labels:
            labels[self.state] = 'A'
        for (r, c), lbl in labels.items():
            ax.text(c, r, lbl, ha='center', va='center', fontsize=14, fontweight='bold')

        if show:
            plt.tight_layout()
            plt.show()


# Verify the environment works
env = GridWorld()
state = env.reset()
next_state, reward, done = env.step(1)  # move Down
print(f"Start: {state}  ->  After 'Down': {next_state}, reward={reward}, done={done}")

## Visualize the Starting State

Before running any agents, look at the map. The agent (A/S) must navigate from
top-left to bottom-right without falling into the hole (H).

In [ ]:
# Purpose: Draw the initial grid so we can reason spatially before writing code
# Key Concept: Visualize the environment before training — know what you're solving

env.reset()
fig, ax = plt.subplots(figsize=(4, 4))
env.render(ax)
ax.set_title('GridWorld: S=Start, G=Goal, H=Hole', fontsize=12)

legend_elements = [
    mpatches.Patch(facecolor='#d73027', label='H (hole, -5)'),
    mpatches.Patch(facecolor='#1a9850', label='G (goal, +10)'),
    mpatches.Patch(facecolor='#fee08b', label='S / A (agent)'),
]
ax.legend(handles=legend_elements, loc='upper right', fontsize=8)
plt.tight_layout()
plt.show()

## Agent 1: Random Policy

The simplest possible agent picks a uniformly random action every step. This is our
baseline — any learning algorithm should beat this.

We run 200 episodes and record the total reward per episode.

In [ ]:
# Purpose: Benchmark the random agent across many episodes
# Key Concept: Episode return = sum of all rewards in one run; we want to maximize this

def run_episodes(env, policy_fn, n_episodes=200):
    """
    Run `n_episodes` episodes using `policy_fn(state) -> action`.
    Returns a list of total rewards (episode returns).
    """
    returns = []
    for _ in range(n_episodes):
        state = env.reset()
        total_reward = 0.0
        done = False
        while not done:
            action = policy_fn(state)
            state, reward, done = env.step(action)
            total_reward += reward
        returns.append(total_reward)
    return returns


# Random policy: ignore state, pick uniformly from {0, 1, 2, 3}
def random_policy(state):
    return np.random.randint(4)


np.random.seed(42)  # reset seed for fair comparison
random_returns = run_episodes(env, random_policy, n_episodes=200)

print(f"Random agent — mean return: {np.mean(random_returns):.2f}  "
      f"std: {np.std(random_returns):.2f}  "
      f"% episodes reaching goal: {np.mean(np.array(random_returns) > 0)*100:.1f}%")

## Agent 2: Greedy Q-Table Agent

We now train a simple Q-table agent using **epsilon-greedy Q-learning**. The Q-table
stores the estimated value of every (state, action) pair. The agent:

1. With probability `epsilon`: explores (random action)
2. With probability `1 - epsilon`: exploits (picks the highest-Q action)

After each step, it updates the Q-value using the **Bellman equation**:

$$Q(s, a) \leftarrow Q(s, a) + \alpha \bigl[r + \gamma \max_{a'} Q(s', a') - Q(s, a)\bigr]$$

Where:
- $\alpha$ (alpha) = learning rate — how quickly to update beliefs
- $\gamma$ (gamma) = discount factor — how much to value future rewards
- $r$ = immediate reward received

In [ ]:
# Purpose: Train a Q-learning agent and record episode returns
# Key Concept: The Bellman update propagates goal reward backwards through state space

class QLearningAgent:
    """
    Tabular Q-learning for discrete state/action spaces.

    Parameters
    ----------
    n_actions : int
        Number of discrete actions available.
    alpha : float
        Learning rate (how aggressively to update Q-values).
    gamma : float
        Discount factor (weight given to future rewards).
    epsilon : float
        Initial exploration probability.
    epsilon_decay : float
        Multiplicative decay applied to epsilon after each episode.
    epsilon_min : float
        Floor on epsilon so the agent never stops exploring entirely.
    """

    def __init__(self, n_actions=4, alpha=0.3, gamma=0.95,
                 epsilon=1.0, epsilon_decay=0.98, epsilon_min=0.05):
        self.n_actions = n_actions
        self.alpha = alpha
        self.gamma = gamma
        self.epsilon = epsilon
        self.epsilon_decay = epsilon_decay
        self.epsilon_min = epsilon_min
        # defaultdict initialises Q(s, a) = 0 for any unseen state
        self.q_table = defaultdict(lambda: np.zeros(n_actions))

    def select_action(self, state):
        """Epsilon-greedy action selection."""
        if np.random.random() < self.epsilon:
            return np.random.randint(self.n_actions)   # explore
        return int(np.argmax(self.q_table[state]))     # exploit

    def update(self, state, action, reward, next_state, done):
        """Apply the Bellman update to Q(state, action)."""
        current_q = self.q_table[state][action]
        if done:
            target = reward                                   # no future if terminal
        else:
            target = reward + self.gamma * np.max(self.q_table[next_state])
        # Move Q-value a fraction alpha toward the target
        self.q_table[state][action] += self.alpha * (target - current_q)

    def decay_epsilon(self):
        """Reduce exploration rate after each episode."""
        self.epsilon = max(self.epsilon_min, self.epsilon * self.epsilon_decay)


def train_q_agent(env, agent, n_episodes=200):
    """Train the agent and return per-episode returns."""
    returns = []
    for _ in range(n_episodes):
        state = env.reset()
        total_reward = 0.0
        done = False
        while not done:
            action = agent.select_action(state)
            next_state, reward, done = env.step(action)
            agent.update(state, action, reward, next_state, done)
            state = next_state
            total_reward += reward
        agent.decay_epsilon()
        returns.append(total_reward)
    return returns


np.random.seed(42)
agent = QLearningAgent(alpha=0.3, gamma=0.95, epsilon=1.0, epsilon_decay=0.98)
q_returns = train_q_agent(env, agent, n_episodes=200)

print(f"Q-learning agent — mean return: {np.mean(q_returns):.2f}  "
      f"% episodes reaching goal: {np.mean(np.array(q_returns) > 0)*100:.1f}%")

## Reward Curves: Random vs Q-Learning

Plot both agents' episode returns side-by-side. A 10-episode rolling average smooths
the noise so the trend is clear.

**What to look for:** The Q-learning curve should trend upward as the agent learns
which actions lead to the goal. The random curve should stay flat and low.

In [ ]:
# Purpose: Compare the two agents visually
# Key Concept: A rising reward curve is the signature of learning

def rolling_mean(data, window=10):
    return np.convolve(data, np.ones(window) / window, mode='valid')


fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# --- Left: raw returns ---
ax = axes[0]
ax.plot(random_returns, color='#e74c3c', alpha=0.4, linewidth=0.8, label='Random (raw)')
ax.plot(q_returns,      color='#2ecc71', alpha=0.4, linewidth=0.8, label='Q-learning (raw)')
ax.plot(rolling_mean(random_returns), color='#c0392b', linewidth=2, label='Random (avg-10)')
ax.plot(rolling_mean(q_returns),      color='#27ae60', linewidth=2, label='Q-learning (avg-10)')
ax.axhline(0, color='grey', linestyle='--', linewidth=0.8)
ax.set_xlabel('Episode', fontsize=12)
ax.set_ylabel('Total Reward', fontsize=12)
ax.set_title('Episode Returns', fontsize=13)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# --- Right: cumulative reward to show overall value gained ---
ax = axes[1]
ax.plot(np.cumsum(random_returns), color='#c0392b', linewidth=2, label='Random')
ax.plot(np.cumsum(q_returns),      color='#27ae60', linewidth=2, label='Q-learning')
ax.set_xlabel('Episode', fontsize=12)
ax.set_ylabel('Cumulative Reward', fontsize=12)
ax.set_title('Cumulative Reward Accumulation', fontsize=13)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.suptitle('Random Policy vs Q-Learning on 4x4 GridWorld', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"Q-learning wins by {np.sum(q_returns) - np.sum(random_returns):.1f} cumulative reward points.")

## Visualize the Learned Policy

After training, we can read the agent's preferred action for every cell from the
Q-table. This is the **greedy policy** — what the agent would do in deployment
with epsilon set to zero.

In [ ]:
# Purpose: Display the learned policy as a grid of arrows
# Key Concept: The Q-table implicitly encodes a complete policy — argmax over actions

ARROW = {0: '↑', 1: '↓', 2: '←', 3: '→'}

fig, ax = plt.subplots(figsize=(5, 5))
ax.set_xlim(-0.5, 3.5)
ax.set_ylim(-0.5, 3.5)
ax.set_xticks(range(4))
ax.set_yticks(range(4))
ax.invert_yaxis()
ax.grid(True, linewidth=1)
ax.set_aspect('equal')

for r in range(4):
    for c in range(4):
        state = (r, c)
        if state == env.goal:
            ax.text(c, r, 'G', ha='center', va='center', fontsize=18,
                    color='green', fontweight='bold')
        elif state == env.hole:
            ax.text(c, r, 'H', ha='center', va='center', fontsize=18,
                    color='red', fontweight='bold')
        elif state in agent.q_table:
            best_action = int(np.argmax(agent.q_table[state]))
            ax.text(c, r, ARROW[best_action], ha='center', va='center', fontsize=20)
        else:
            ax.text(c, r, '?', ha='center', va='center', fontsize=14, color='grey')

ax.set_title('Learned Greedy Policy (arrows = best action per cell)', fontsize=12)
plt.tight_layout()
plt.show()

## Modify and Experiment

Try these modifications to build intuition:

1. **Change `alpha` to 0.01** — does the agent learn more slowly?
2. **Change `gamma` to 0.5** — the agent discounts the future heavily. Does it still
   find the goal reliably?
3. **Change `epsilon_decay` to 0.99** — slower exploration decay. Does the agent
   take longer to converge?
4. **Move the hole to `(3, 2)`** — what does the new learned policy look like?

Each experiment takes less than five seconds to rerun.

## Key Takeaways

1. **The RL loop** is always: observe state → choose action → receive reward → update
2. **A random baseline** is cheap to compute and tells you the floor performance
3. **Q-learning** stores value estimates in a table and improves them via the Bellman update
4. **Epsilon-greedy** balances exploration (trying new things) with exploitation (using
   what you know)
5. **Rising reward curves** confirm that learning is happening

## What's Next

This gridworld has only 16 states, so a Q-table fits in memory easily. When states
are continuous (like cart position and velocity in CartPole), a table becomes
impractical. Head to **Quick-Start 01** to see how discretization extends Q-learning
to CartPole-v1, and then **Quick-Start 02** to replace the table entirely with a
neural network.

For deeper coverage, see `modules/module_02_q_learning/` in this course.